In [9]:
"""
PYSTAC-Monty CSV Extractor

This script extracts data from PYSTAC-Monty transformers.
Requires pystac-monty libraries to be installed and accessible.
"""

import argparse
import csv
import json
import logging
import os
import sys
import requests
from typing import Dict, List, Any, Optional, Tuple, Type

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Define a function to check and import required modules
def import_pystac_monty_modules():
    # Check if pystac is installed
    try:
        import pystac
    except ImportError:
        logger.error("Failed to import pystac module")
        logger.error("Please make sure pystac library is installed")
        sys.exit(1)
        
    # Check if pystac_monty is installed
    try:
        import pystac_monty
    except ImportError:
        logger.error("Failed to import pystac_monty module")
        logger.error("Please make sure pystac-monty library is installed")
        sys.exit(1)
    
    # Now import all required modules
    global Item, MontyDataTransformer, MontyDataSource, MontyGeoCoder, MontyExtension, MontyHazardProfiles
    global EMDATTransformer, EMDATDataSource, DesinventarTransformer, DesinventarDataSource
    global GDACSTransformer, GDACSDataSource, GDACSDataSourceType, GFDTransformer, GFDDataSource
    global GIDDTransformer, GIDDDataSource, GlideTransformer, GlideDataSource
    global IBTrACSTransformer, IBTrACSDataSource, IDUTransformer, IDUDataSource
    global PDCTransformer, PDCDataSource, USGSTransformer, USGSDataSource
    
    from pystac import Item
    from pystac_monty.sources.common import MontyDataTransformer, MontyDataSource
    from pystac_monty.geocoding import MontyGeoCoder
    from pystac_monty.extension import MontyExtension
    from pystac_monty.hazard_profiles import MontyHazardProfiles

    # Import all transformers and data sources
    from pystac_monty.sources.emdat import EMDATTransformer, EMDATDataSource
    from pystac_monty.sources.desinventar import DesinventarTransformer, DesinventarDataSource
    from pystac_monty.sources.gdacs import GDACSTransformer, GDACSDataSource
    from pystac_monty.sources.gfd import GFDTransformer, GFDDataSource
    from pystac_monty.sources.gidd import GIDDTransformer, GIDDDataSource
    from pystac_monty.sources.glide import GlideTransformer, GlideDataSource
    from pystac_monty.sources.ibtracs import IBTrACSTransformer, IBTrACSDataSource
    from pystac_monty.sources.idu import IDUTransformer, IDUDataSource
    #from pystac_monty.sources.ifrcevent import IFRCEventTransformer, IFRCEventDataSource
    from pystac_monty.sources.pdc import PDCTransformer, PDCDataSource
    from pystac_monty.sources.usgs import USGSTransformer, USGSDataSource

# Call the import function
import_pystac_monty_modules()


class StacItemExtractor:
    """Extract data from STAC Items produced by MontyDataTransformers."""
    
    def __init__(self, output_dir: str = "output"):
        """Initialize the extractor.
        
        Args:
            output_dir: Directory where CSV files will be saved
        """
        self.output_dir = output_dir
        os.makedirs(output_dir, exist_ok=True)
        
        # Initialize data containers
        self.events: List[Dict[str, Any]] = []
        self.hazards: List[Dict[str, Any]] = []
        self.impacts: List[Dict[str, Any]] = []
        
        # Track relationships between items
        self.event_map: Dict[str, str] = {}  # Maps hazard/impact to event
        self.hazard_map: Dict[str, str] = {}  # Maps impact to hazard
    
    def extract_from_transformer(self, transformer: MontyDataTransformer, source_name: str) -> None:
        """Extract data from a transformer and add it to the collection.
        
        Args:
            transformer: The MontyDataTransformer to extract data from
            source_name: Name of the data source
        """
        logger.info(f"Extracting data from {source_name} transformer")
        
        # Process items from the transformer
        current_event_id = None
        current_hazard_id = None
        
        # Different transformers have different methods to get items
        # Try different possible methods
        items = []
        
        try:
            # First try get_stac_items method if it exists
            if hasattr(transformer, 'get_stac_items'):
                items = transformer.get_stac_items()
                logger.info(f"Used get_stac_items() method for {source_name}")
            # Try transform_to_stac_items if get_stac_items doesn't exist
            elif hasattr(transformer, 'transform_to_stac_items'):
                items = transformer.transform_to_stac_items()
                logger.info(f"Used transform_to_stac_items() method for {source_name}")
            # Try get_items if neither exists
            elif hasattr(transformer, 'get_items'):
                items = transformer.get_items()
                logger.info(f"Used get_items() method for {source_name}")
            # Try transform method as a last resort
            elif hasattr(transformer, 'transform'):
                items = transformer.transform()
                logger.info(f"Used transform() method for {source_name}")
            # If no suitable method is found, try calling the transformer directly
            elif callable(transformer):
                items = transformer()
                logger.info(f"Called transformer directly for {source_name}")
            else:
                # No method available to retrieve items
                logger.error(f"No item retrieval method found for {source_name} transformer")
                return
        except Exception as e:
            logger.error(f"Error retrieving items from transformer: {e}")
            return
        
        # Log the number of items found
        if items:
            if isinstance(items, (list, tuple)):
                logger.info(f"Retrieved {len(items)} items from {source_name}")
            else:
                logger.info(f"Retrieved 1 item from {source_name}")
        else:
            logger.warning(f"No items retrieved from {source_name}")
            return
        
        # Ensure items is iterable even if a single item was returned
        if not isinstance(items, (list, tuple)):
            items = [items]
        
        for item in items:
            if not isinstance(item, Item):
                logger.warning(f"Non-Item object found: {type(item)}, skipping")
                continue
                
            roles = item.properties.get("roles", [])
            
            if "event" in roles:
                # Store event and set as current event
                event_data = self._extract_event_data(item, source_name)
                self.events.append(event_data)
                current_event_id = item.id
                logger.info(f"Added event item: {item.id}")
                
            elif "hazard" in roles:
                # Store hazard and link to current event
                if current_event_id:
                    self.event_map[item.id] = current_event_id
                
                hazard_data = self._extract_hazard_data(item, source_name)
                self.hazards.append(hazard_data)
                current_hazard_id = item.id
                logger.info(f"Added hazard item: {item.id}")
                
            elif "impact" in roles:
                # Store impact and link to current event and hazard
                if current_event_id:
                    self.event_map[item.id] = current_event_id
                if current_hazard_id:
                    self.hazard_map[item.id] = current_hazard_id
                
                impact_data = self._extract_impact_data(item, source_name)
                self.impacts.append(impact_data)
                logger.info(f"Added impact item: {item.id}")
            else:
                logger.warning(f"Item {item.id} has no recognized role in {roles}, skipping")
    
    def _extract_event_data(self, item: Item, source: str) -> Dict[str, Any]:
        """Extract data from an event item.
        
        Args:
            item: STAC Item representing an event
            source: Name of the data source
            
        Returns:
            Dictionary of extracted event data
        """
        # Extract MontyExtension data
        monty = MontyExtension.ext(item)
        country_codes = monty.country_codes if hasattr(monty, "country_codes") else []
        hazard_codes = monty.hazard_codes if hasattr(monty, "hazard_codes") else []
        correlation_id = monty.correlation_id if hasattr(monty, "correlation_id") else None
        
        # Get start and end datetimes
        start_datetime = item.properties.get("start_datetime", None)
        end_datetime = item.properties.get("end_datetime", None)
        
        # Extract event data
        event_data = {
            "id": item.id,
            "source": source,
            "title": item.properties.get("title", ""),
            "description": item.properties.get("description", ""),
            "datetime": item.datetime.isoformat() if item.datetime else None,
            "start_datetime": start_datetime,
            "end_datetime": end_datetime,
            "country_codes": "|".join(country_codes) if country_codes else "",
            "hazard_codes": "|".join(hazard_codes) if hazard_codes else "",
            "correlation_id": correlation_id,
            "bbox": json.dumps(item.bbox) if item.bbox else "",
            "geometry": json.dumps(item.geometry) if item.geometry else ""
        }
        
        return event_data
    
    def _extract_hazard_data(self, item: Item, source: str) -> Dict[str, Any]:
        """Extract data from a hazard item.
        
        Args:
            item: STAC Item representing a hazard
            source: Name of the data source
            
        Returns:
            Dictionary of extracted hazard data
        """
        # Extract MontyExtension data
        monty = MontyExtension.ext(item)
        country_codes = monty.country_codes if hasattr(monty, "country_codes") else []
        hazard_codes = monty.hazard_codes if hasattr(monty, "hazard_codes") else []
        correlation_id = monty.correlation_id if hasattr(monty, "correlation_id") else None
        
        # Extract hazard detail
        hazard_detail = monty.hazard_detail if hasattr(monty, "hazard_detail") else None
        severity_value = None
        severity_unit = None
        severity_label = None
        if hazard_detail:
            severity_value = hazard_detail.severity_value if hasattr(hazard_detail, "severity_value") else None
            severity_unit = hazard_detail.severity_unit if hasattr(hazard_detail, "severity_unit") else None
            severity_label = hazard_detail.severity_label if hasattr(hazard_detail, "severity_label") else None
        
        # Get linked event_id
        event_id = self.event_map.get(item.id, "")
        
        # Extract hazard data
        hazard_data = {
            "id": item.id,
            "event_id": event_id,
            "source": source,
            "title": item.properties.get("title", ""),
            "description": item.properties.get("description", ""),
            "datetime": item.datetime.isoformat() if item.datetime else None,
            "country_codes": "|".join(country_codes) if country_codes else "",
            "hazard_codes": "|".join(hazard_codes) if hazard_codes else "",
            "correlation_id": correlation_id,
            "severity_value": severity_value,
            "severity_unit": severity_unit,
            "severity_label": severity_label,
            "bbox": json.dumps(item.bbox) if item.bbox else "",
            "geometry": json.dumps(item.geometry) if item.geometry else ""
        }
        
        return hazard_data
    
    def _extract_impact_data(self, item: Item, source: str) -> Dict[str, Any]:
        """Extract data from an impact item.
        
        Args:
            item: STAC Item representing an impact
            source: Name of the data source
            
        Returns:
            Dictionary of extracted impact data
        """
        # Extract MontyExtension data
        monty = MontyExtension.ext(item)
        country_codes = monty.country_codes if hasattr(monty, "country_codes") else []
        correlation_id = monty.correlation_id if hasattr(monty, "correlation_id") else None
        
        # Extract impact detail
        impact_detail = monty.impact_detail if hasattr(monty, "impact_detail") else None
        impact_category = None
        impact_type = None
        value = None
        unit = None
        if impact_detail:
            impact_category = impact_detail.category if hasattr(impact_detail, "category") else None
            impact_type = impact_detail.type if hasattr(impact_detail, "type") else None
            value = impact_detail.value if hasattr(impact_detail, "value") else None
            unit = impact_detail.unit if hasattr(impact_detail, "unit") else None
        
        # Get linked event_id and hazard_id
        event_id = self.event_map.get(item.id, "")
        hazard_id = self.hazard_map.get(item.id, "")
        
        # Extract impact data
        impact_data = {
            "id": item.id,
            "event_id": event_id,
            "hazard_id": hazard_id,
            "source": source,
            "title": item.properties.get("title", ""),
            "description": item.properties.get("description", ""),
            "datetime": item.datetime.isoformat() if item.datetime else None,
            "country_codes": "|".join(country_codes) if country_codes else "",
            "correlation_id": correlation_id,
            "impact_category": str(impact_category) if impact_category else "",
            "impact_type": str(impact_type) if impact_type else "",
            "value": value,
            "unit": unit,
            "bbox": json.dumps(item.bbox) if item.bbox else "",
            "geometry": json.dumps(item.geometry) if item.geometry else ""
        }
        
        return impact_data
    
    def save_to_csv(self) -> None:
        """Save extracted data to CSV files."""
        # Save events
        if self.events:
            self._save_list_to_csv(self.events, os.path.join(self.output_dir, "events.csv"))
        
        # Save hazards
        if self.hazards:
            self._save_list_to_csv(self.hazards, os.path.join(self.output_dir, "hazards.csv"))
        
        # Save impacts
        if self.impacts:
            self._save_list_to_csv(self.impacts, os.path.join(self.output_dir, "impacts.csv"))
    
    def _save_list_to_csv(self, data_list: List[Dict[str, Any]], output_path: str) -> None:
        """Save a list of dictionaries to a CSV file.
        
        Args:
            data_list: List of dictionaries to save
            output_path: Path to save the CSV file
        """
        if not data_list:
            logger.warning(f"No data to save to {output_path}")
            return
        
        # Get fieldnames from the first item
        fieldnames = list(data_list[0].keys())
        
        logger.info(f"Saving {len(data_list)} records to {output_path}")
        
        with open(output_path, 'w', newline='', encoding='utf-8') as csvfile:
            writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(data_list)


# Define real data configuration for East African flood and drought data
# Using shorter query params to avoid filename length issues
SOURCE_CONFIGS = {
    "emdat": {
        "source_url": "https://public.emdat.be/data",
        # Use a simpler query to avoid filename length errors
        "data_params": {
            "iso": ["KEN", "ETH", "UGA", "TZA", "SOM", "SSD"],
            "hazard_type": ["Drought", "Flood"],
            "from_year": 2020,
            "to_year": 2025,
            "limit": 50
        }
    },
    "desinventar": {
        "tmp_zip_file": "data/temp/desinventar.zip",
        "country_code": "ken",  # Kenya
        "iso3": "KEN"
    },
    "gdacs_flood": {
        "source_url": "https://www.gdacs.org/gdacsapi/api/events",
        "data_params": {
            "eventtype": "FL",
            "iso3": ["KEN", "ETH", "UGA", "TZA", "SOM", "SSD"],
            "fromdate": "2023-01-01",
            "todate": "2025-04-23"
        },
        "type": "FL"
    },
    "gdacs_drought": {
        "source_url": "https://www.gdacs.org/gdacsapi/api/events",
        "data_params": {
            "eventtype": "DR",
            "iso3": ["KEN", "ETH", "UGA", "TZA", "SOM", "SSD"],
            "fromdate": "2023-01-01",
            "todate": "2025-04-23"
        },
        "type": "DR"
    },
    "gfd": {
        "source_url": "https://global-flood-database.cloudtostreet.ai/",
        "data_params": {
            "region": "East Africa",
            "countries": ["KEN", "ETH", "UGA", "TZA", "SOM", "SSD"],
            "start_date": "2023-01-01",
            "end_date": "2025-04-23"
        }
    },
    "gidd": {
        "source_url": "https://helix-tools-api.idmcdb.org/external-api/gidd/disaggregations/disaggregation-geojson/",
        "data_params": {
            "iso": ["KEN", "ETH", "UGA", "TZA", "SOM", "SSD"],
            "hazard_type": ["Flood", "Drought"],
            "from": 2020,
            "to": 2025
        }
    },
    "glide": {
        "source_url": "https://www.glidenumber.net/glide/jsonglideset.jsp",
        "data_params": {
            "level1": ["KEN", "ETH", "UGA", "TZA", "SOM", "SSD"],
            "events": ["FL", "DR"],
            "fromyear": "2020",
            "toyear": "2025"
        }
    },
    "ibtracs": {
        "source_url": "https://www.ncei.noaa.gov/data/international-best-track-archive-for-climate-stewardship-ibtracs/v04r01/access/csv/",
        "data_params": {
            "basin": "IO", # Indian Ocean basin
            "year": "2023"
        }
    },
    "idu": {
        "source_url": "https://helix-tools-api.idmcdb.org/external-api/",
        "data_params": {
            "iso3": ["KEN", "ETH", "UGA", "TZA", "SOM", "SSD"],
            "hazard_type": ["Flood", "Drought"],
            "from_date": "2020-01-01",
            "to_date": "2025-04-23"
        }
    },
    "pdc": {
        "source_url": "https://sentry.pdc.org/hp_srv/services/hazards/",
        "data_params": {
            "countryIso3": ["KEN", "ETH", "UGA", "TZA", "SOM", "SSD"],
            "hazardType": ["flood", "drought"],
            "startDate": "2020-01-01",
            "endDate": "2025-04-23"
        }
    },
    "usgs": {
        "source_url": "https://earthquake.usgs.gov/earthquakes/feed/v1.0/summary/",
        "data_params": {
            "format": "geojson",
            "starttime": "2023-01-01",
            "endtime": "2025-04-23",
            "minlatitude": -12,
            "maxlatitude": 23,
            "minlongitude": 25,
            "maxlongitude": 52,
            "minmagnitude": 4.5
        }
    }
}


# Define mapping of source names to transformer/data source classes
SOURCE_MAPPING = {
    "emdat": (EMDATTransformer, EMDATDataSource),
    "desinventar": (DesinventarTransformer, DesinventarDataSource),
    "gdacs_flood": (GDACSTransformer, GDACSDataSource),
    "gdacs_drought": (GDACSTransformer, GDACSDataSource),
    "gfd": (GFDTransformer, GFDDataSource),
    "gidd": (GIDDTransformer, GIDDDataSource),
    "glide": (GlideTransformer, GlideDataSource),
    "ibtracs": (IBTrACSTransformer, IBTrACSDataSource),
    "idu": (IDUTransformer, IDUDataSource),
    #"ifrcevent": (IFRCEventTransformer, IFRCEventDataSource),
    "pdc": (PDCTransformer, PDCDataSource),
    "usgs": (USGSTransformer, USGSDataSource),
}


def setup_logging(level: int = logging.INFO, log_file: Optional[str] = None) -> None:
    """Set up logging configuration.
    
    Args:
        level: Logging level (default: logging.INFO)
        log_file: Path to log file (if None, log to stdout only)
    """
    handlers = [logging.StreamHandler(sys.stdout)]
    
    if log_file:
        handlers.append(logging.FileHandler(log_file))
    
    logging.basicConfig(
        level=level,
        format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
        handlers=handlers
    )


def create_parser() -> argparse.ArgumentParser:
    """Create an argument parser for the CLI.
    
    Returns:
        An argument parser for the CLI
    """
    parser = argparse.ArgumentParser(
        description="Extract data from PYSTAC-Monty transformers and output as CSV"
    )
    
    subparsers = parser.add_subparsers(dest="command", help="Command to run")
    
    # Extract command
    extract_parser = subparsers.add_parser("extract", help="Extract data from transformers")
    extract_parser.add_argument(
        "--source", "-s", 
        required=True,
        help="Source(s) to extract data from, comma-separated if multiple"
    )
    extract_parser.add_argument(
        "--output", "-o",
        default="output",
        help="Directory to save output CSV files (default: output)"
    )
    extract_parser.add_argument(
        "--verbose", "-v",
        action="store_true",
        help="Enable verbose logging"
    )
    extract_parser.add_argument(
        "--log-file",
        help="Path to save log file"
    )
    
    # List command (to list available sources)
    list_parser = subparsers.add_parser("list", help="List available sources")
    
    return parser


def main_extract_workflow():
    """Run a simplified extraction workflow that focuses on debugging issues."""
    print("Starting simplified extraction workflow for ICPAC region disaster data...")
    
    # Configure logging
    logging.basicConfig(level=logging.INFO)
    
    # Initialize the ICPAC geocoder
    geocoder = ICPACGeoCoder()
    print(f"Initialized ICPAC geocoder with {len(geocoder.icpac_countries)} countries")
    
    # Create output directory
    output_dir = "output"
    os.makedirs(output_dir, exist_ok=True)
    
    # Track extracted data
    events = []
    hazards = []
    impacts = []
    
    # Try a simplified test with USGS earthquake data for East Africa
    try:
        print("Testing USGS earthquake data extraction for East Africa...")
        
        # USGS query parameters for East Africa
        params = {
            "format": "geojson",
            "starttime": "2023-01-01",
            "endtime": "2025-04-23",
            "minlatitude": -12,
            "maxlatitude": 23,
            "minlongitude": 25,
            "maxlongitude": 52,
            "minmagnitude": 4.5
        }
        
        # Build the API URL with query parameters
        base_url = "https://earthquake.usgs.gov/fdsnws/event/1/query"
        query_params = "&".join([f"{key}={value}" for key, value in params.items()])
        api_url = f"{base_url}?{query_params}"
        
        print(f"Fetching earthquake data from: {api_url}")
        
        # Fetch the data
        response = requests.get(api_url)
        
        if response.status_code == 200:
            earthquake_data = response.json()
            
            # Process the GeoJSON features
            features = earthquake_data.get('features', [])
            print(f"Retrieved {len(features)} earthquake events")
            
            # Extract the events
            for feature in features:
                # Basic properties
                event_id = feature['id']
                geometry = feature['geometry']
                properties = feature['properties']
                
                # Create a simplified event record
                event = {
                    'id': event_id,
                    'source': 'usgs',
                    'title': properties.get('title', ''),
                    'datetime': properties.get('time'),
                    'magnitude': properties.get('mag'),
                    'place': properties.get('place', ''),
                    'type': properties.get('type', ''),
                    'longitude': geometry['coordinates'][0],
                    'latitude': geometry['coordinates'][1],
                    'depth': geometry['coordinates'][2],
                }
                
                # Determine which ICPAC country this is in
                iso3 = geocoder.get_iso3_from_geometry(geometry)
                if iso3:
                    event['country_code'] = iso3
                
                events.append(event)
            
            # Save to CSV
            if events:
                with open(os.path.join(output_dir, "usgs_events.csv"), 'w', newline='') as csvfile:
                    fieldnames = events[0].keys()
                    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
                    writer.writeheader()
                    writer.writerows(events)
                print(f"Saved {len(events)} USGS earthquake events to CSV")
        else:
            print(f"Failed to fetch USGS data: {response.status_code}")
    
    except Exception as e:
        print(f"Error processing USGS data: {e}")
    
    # Try a simplified test with GDACS flood data for East Africa
    try:
        print("\nTesting GDACS flood data extraction for East Africa...")
        
        # GDACS API endpoint for floods
        gdacs_url = "https://www.gdacs.org/gdacsapi/api/events/geteventlist/SEARCH"
        
        # Parameters for East African floods
        params = {
            "eventtype": "FL",
            "from": "2023-01-01",
            "to": "2025-04-23",
            "country": "Kenya,Ethiopia,Uganda,Tanzania,Somalia,South Sudan"
        }
        
        print(f"Fetching flood data from GDACS API")
        
        # Fetch the data
        response = requests.get(gdacs_url, params=params)
        
        if response.status_code == 200:
            flood_data = response.json()
            
            # Process the features
            features = flood_data.get('features', [])
            print(f"Retrieved {len(features)} flood events")
            
            # Extract the events
            for feature in features:
                # Basic properties
                properties = feature.get('properties', {})
                geometry = feature.get('geometry', {})
                
                # Create a simplified event record
                event = {
                    'id': properties.get('eventid'),
                    'source': 'gdacs',
                    'title': properties.get('name', ''),
                    'description': properties.get('description', ''),
                    'fromdate': properties.get('fromdate'),
                    'todate': properties.get('todate'),
                    'country': properties.get('country', ''),
                    'iso3': properties.get('iso3', ''),
                    'alertlevel': properties.get('alertlevel'),
                    'alertscore': properties.get('alertscore'),
                }
                
                if geometry and 'coordinates' in geometry:
                    event['longitude'] = geometry['coordinates'][0]
                    event['latitude'] = geometry['coordinates'][1]
                
                events.append(event)
            
            # Save to CSV
            if events:
                with open(os.path.join(output_dir, "gdacs_floods.csv"), 'w', newline='') as csvfile:
                    fieldnames = events[0].keys()
                    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
                    writer.writeheader()
                    writer.writerows(events)
                print(f"Saved {len(events)} GDACS flood events to CSV")
        else:
            print(f"Failed to fetch GDACS data: {response.status_code}")
    
    except Exception as e:
        print(f"Error processing GDACS data: {e}")
    
    print("\nSimplified extraction workflow complete!")


# Simple implementation of ICPACGeoCoder
class ICPACGeoCoder:
    """A geocoder implementation focusing on ICPAC regions (East Africa)."""
    
    def __init__(self):
        """Initialize the ICPAC geocoder with regions data."""
        # ICPAC member countries (Greater Horn of Africa)
        self.icpac_countries = {
            "DJI": {"name": "Djibouti", "center": [42.590275, 11.825138]},
            "ERI": {"name": "Eritrea", "center": [39.782334, 15.179384]},
            "ETH": {"name": "Ethiopia", "center": [40.489673, 9.145000]},
            "KEN": {"name": "Kenya", "center": [37.906193, 0.023559]},
            "SOM": {"name": "Somalia", "center": [46.199616, 5.152149]},
            "SSD": {"name": "South Sudan", "center": [31.306978, 6.876991]},
            "SDN": {"name": "Sudan", "center": [30.217636, 12.862807]},
            "UGA": {"name": "Uganda", "center": [32.290275, 1.373333]},
            "BDI": {"name": "Burundi", "center": [29.918886, -3.373056]},
            "RWA": {"name": "Rwanda", "center": [29.873888, -1.940278]},
            "TZA": {"name": "Tanzania", "center": [34.888822, -6.369028]}
        }
        
        # Country bounding boxes (rough estimates for simplicity)
        self.country_bounds = {
            "DJI": [41.7, 10.9, 43.4, 12.7],  # [min_lon, min_lat, max_lon, max_lat]
            "ERI": [36.4, 12.3, 43.1, 18.0],
            "ETH": [33.0, 3.4, 48.0, 14.9],
            "KEN": [33.9, -4.7, 41.9, 5.0],
            "SOM": [40.0, -1.6, 51.4, 11.3],
            "SSD": [24.1, 3.5, 36.0, 12.2],
            "SDN": [21.8, 8.7, 38.6, 22.2],
            "UGA": [29.5, -1.5, 35.0, 4.2],
            "BDI": [29.0, -4.5, 30.8, -2.3],
            "RWA": [28.8, -2.8, 30.9, -1.0],
            "TZA": [29.3, -11.7, 40.4, -1.0]
        }
    
    def get_geometry_by_country_name(self, country_name):
        """Get geometry for a country by name."""
        for iso3, data in self.icpac_countries.items():
            if data["name"].lower() == country_name.lower():
                # Return a point geometry for the country's center
                return {"type": "Point", "coordinates": data["center"]}
        
        # Default
        return {"type": "Point", "coordinates": [37.0, 3.0]}
    
    def get_geometry_from_admin_units(self, admin_units):
        """Get geometry from a list of admin units."""
        # Simplified - returns center of Kenya
        return {"type": "Point", "coordinates": self.icpac_countries["KEN"]["center"]}
    
    def get_geometry_from_iso3(self, iso3):
        """Get geometry for a country by ISO3 code."""
        if iso3 in self.icpac_countries:
            return {"type": "Point", "coordinates": self.icpac_countries[iso3]["center"]}
        
        # Default
        return {"type": "Point", "coordinates": [37.0, 3.0]}
    
    def get_iso3_from_geometry(self, geometry):
        """Get ISO3 code for a country based on a geometry."""
        if geometry and geometry.get("type") == "Point":
            coords = geometry.get("coordinates", [0, 0])
            lon, lat = coords[0], coords[1]
            
            # Check which country bounds the point falls within
            for iso3, bounds in self.country_bounds.items():
                min_lon, min_lat, max_lon, max_lat = bounds
                if min_lon <= lon <= max_lon and min_lat <= lat <= max_lat:
                    return iso3
            
            # If not in any bounds, find closest country center
            closest_iso3 = None
            min_distance = float('inf')
            
            for iso3, data in self.icpac_countries.items():
                country_coords = data["center"]
                distance = ((lon - country_coords[0])**2 + (lat - country_coords[1])**2)**0.5
                
                if distance < min_distance:
                    min_distance = distance
                    closest_iso3 = iso3
            
            if closest_iso3:
                return closest_iso3
        
        # Default to Kenya
        return "KEN"


# Execute the simplified workflow when running in a Jupyter notebook
if 'ipykernel' in sys.modules:
    main_extract_workflow()


def list_command() -> None:
    """Execute the list command."""
    print("Available sources:")
    for source in sorted(SOURCE_MAPPING.keys()):
        print(f"  - {source}")


def main() -> None:
    """Main entry point for the CLI."""
    # Check if running in a Jupyter notebook
    is_jupyter = 'ipykernel' in sys.modules
    
    if is_jupyter:
        # If in Jupyter, skip argparse and extract from all sources
        print("Running in Jupyter notebook environment")
        print("Extracting data from all available sources...")
        
        # Get all available sources
        sources = list(SOURCE_MAPPING.keys())
        print(f"Sources to process: {', '.join(sources)}")
        
        # Extract data from all sources
        output_dir = "output"
        extract_command(sources=sources, output_dir=output_dir, verbose=True)
    else:
        # Normal CLI execution
        parser = create_parser()
        args = parser.parse_args()
        
        if args.command == "extract":
            sources = [s.strip() for s in args.source.split(",")]
            extract_command(
                sources=sources,
                output_dir=args.output,
                verbose=args.verbose,
                log_file=args.log_file
            )
        elif args.command == "list":
            list_command()
        else:
            parser.print_help()
            sys.exit(1)


if __name__ == "__main__":
    main()

Starting simplified extraction workflow for ICPAC region disaster data...
Initialized ICPAC geocoder with 11 countries
Testing USGS earthquake data extraction for East Africa...
Fetching earthquake data from: https://earthquake.usgs.gov/fdsnws/event/1/query?format=geojson&starttime=2023-01-01&endtime=2025-04-23&minlatitude=-12&maxlatitude=23&minlongitude=25&maxlongitude=52&minmagnitude=4.5
Retrieved 270 earthquake events
Saved 270 USGS earthquake events to CSV

Testing GDACS flood data extraction for East Africa...
Fetching flood data from GDACS API


2025-04-23 07:29:11,859 - __main__ - INFO - Initialized extractor with output directory: output
2025-04-23 07:29:11,860 - __main__ - INFO - Initialized ICPAC geocoder with 11 countries
2025-04-23 07:29:11,860 - __main__ - INFO - Processing source: emdat
2025-04-23 07:29:11,861 - __main__ - ERROR - Error processing emdat: [Errno 2] No such file or directory: '{"iso": ["KEN", "ETH", "UGA", "TZA", "SOM", "SSD"], "hazard_type": ["Drought", "Flood"], "from_year": 2020, "to_year": 2025, "limit": 50}'
2025-04-23 07:29:11,861 - __main__ - INFO - Processing source: desinventar
2025-04-23 07:29:11,862 - __main__ - INFO - Extracting data from desinventar transformer
2025-04-23 07:29:11,862 - __main__ - ERROR - Error retrieving items from transformer: generator didn't yield
2025-04-23 07:29:11,863 - __main__ - INFO - Processing source: gdacs_flood
2025-04-23 07:29:11,864 - __main__ - INFO - Extracting data from gdacs_flood transformer
2025-04-23 07:29:11,864 - __main__ - ERROR - No item retrieval 

Failed to fetch GDACS data: 404

Simplified extraction workflow complete!
Running in Jupyter notebook environment
Extracting data from all available sources...
Sources to process: emdat, desinventar, gdacs_flood, gdacs_drought, gfd, gidd, glide, ibtracs, idu, pdc, usgs
